# chars

> Extract character-level features, like character ngrams. 

This functionality is not available in the latest version available on Pypi (0.0.8), but will be released as part of version 0.0.9.

In [ ]:
#| default_exp chars

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from sklearn.base import BaseEstimator, TransformerMixin
from fastcore.basics import patch
from textplumber.store import TextFeatureStore
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
#| export
class CharNgramVectorizer(BaseEstimator, TransformerMixin):
	""" Sci-kit Learn pipeline component to extract character ngram features. """
	def __init__(self, 
			  feature_store:TextFeatureStore = None, # (not implemented currently)
			  vectorizer_type:str = 'count', # the type of vectorizer to use - 'count' for CountVectorizer or 'tfidf' for TfidfVectorizer
			  ngram_range:tuple = (2, 2), # the ngram range to use (min_n, max_n) - passed to CountVectorizer or TfidfVectorizer
			  lowercase:bool = False, # whether to lowercase the character ngrams - passed to CountVectorizer or TfidfVectorizer 
			  min_df:float|int = 1, # the minimum document frequency to use - passed to CountVectorizer or TfidfVectorizer
			  max_df:float|int = 1.0, # the maximum document frequency to use - passed to CountVectorizer or TfidfVectorizer
			  max_features:int = 5000, # the maximum number of features to use, setting a default to avoid memory issues - passed to CountVectorizer or TfidfVectorizer
			  vocabulary:list|None = None, # list of tokens to use - passed to CountVectorizer or TfidfVectorizer
			  analyzer:str = 'char', # the analyzer to use - 'char' or 'char_wb - passed to CountVectorizer or TfidfVectorizer
			  encoding:str = 'utf-8', # the encoding to use - passed to CountVectorizer or TfidfVectorizer 
			  decode_error:str = 'ignore' # what to do if there is an error decoding 'strict', 'ignore', 'replace' - passed to CountVectorizer or TfidfVectorizer
			  ):
		
		self.feature_store = feature_store
		self.vectorizer_type = vectorizer_type
		self.ngram_range = ngram_range
		self.lowercase = lowercase
		self.min_df = min_df
		self.max_df = max_df
		self.max_features = max_features
		self.vocabulary = vocabulary
		self.analyzer = analyzer
		self.encoding = encoding
		self.decode_error = decode_error


In [ ]:
#| export
@patch
def fit(self:CharNgramVectorizer, X, y=None):
	""" Fit the vectorizer. """
	if self.vectorizer_type == 'tfidf':
		self.vectorizer_ = TfidfVectorizer(analyzer = self.analyzer, lowercase=self.lowercase, min_df=self.min_df, max_df=self.max_df, max_features=self.max_features, ngram_range=self.ngram_range, vocabulary= self.vocabulary, encoding=self.encoding, decode_error=self.decode_error)
	elif self.vectorizer_type == 'count':
		self.vectorizer_ = CountVectorizer(analyzer = self.analyzer, lowercase=self.lowercase, min_df=self.min_df, max_df=self.max_df, max_features=self.max_features, ngram_range=self.ngram_range, vocabulary= self.vocabulary, encoding=self.encoding, decode_error=self.decode_error)
	else:
		raise ValueError("Invalid vectorizer_type. Use 'tfidf' or 'count'.")
	self.vectorizer_.fit(X, y)
	return self

In [ ]:
#| export
@patch
def transform(self:CharNgramVectorizer, X):
	""" Transform the texts to a matrix of counts or tf-idf scores. """
	return self.vectorizer_.transform(X)



In [ ]:
#| export
@patch
def get_feature_names_out(self:CharNgramVectorizer, input_features=None):
	""" Get the feature names out from the model. """
	return self.vectorizer_.get_feature_names_out(input_features)

In [ ]:
#| hide
char_ngram_vectorizer = CharNgramVectorizer()
char_ngram_vectorizer.fit(['Hello, world! Hello, universe!'])
X = char_ngram_vectorizer.fit_transform(['Hello, world! Hello, universe!'])
id = char_ngram_vectorizer.get_feature_names_out().tolist().index(' H')
assert X.todense()[0, id] == 1
id = char_ngram_vectorizer.get_feature_names_out().tolist().index('ll')
assert X.todense()[0, id] == 2

## Example

Here is an example demonstrating how to use `CharNgramVectorizer` in a pipeline.

In [ ]:
#| eval: false
from textplumber.chars import CharNgramVectorizer
from textplumber.core import get_example_data
from textplumber.report import plot_confusion_matrix

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2

Here we load text samples from Ernest Hemingway and Virginia Woolf available in the [AuthorMix dataset](https://huggingface.co/datasets/hallisky/AuthorMix).

In [ ]:
#| eval: false
X_train, y_train, X_test, y_test, target_classes, target_names = get_example_data(label_column = 'style', target_labels = ['hemingway', 'woolf'])

The next cell creates a very simply classification pipeline that extracts 1000 lower-cased character bigrams as features.

In [ ]:
#| eval: false
pipeline = Pipeline([
    ('charngrams', CharNgramVectorizer(ngram_range = (2, 2), lowercase = True, max_features=1000)),
    ('classifier', LogisticRegression(max_iter = 5000, random_state=55))
], verbose=True)

display(pipeline)

Pipeline(steps=[('charngrams', CharNgramVectorizer(max_features=1000)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=55))],
         verbose=True)

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ........ (step 1 of 2) Processing charngrams, total=   0.8s
[Pipeline] ........ (step 2 of 2) Processing classifier, total=   0.9s


In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

   hemingway      0.921     0.921     0.921       504
       woolf      0.918     0.918     0.918       488

    accuracy                          0.919       992
   macro avg      0.919     0.919     0.919       992
weighted avg      0.919     0.919     0.919       992



The `lowercase` is set to False by default, meaning 'Go' is different to 'go'. As this example shows, preserving case can make a difference to accuracy.

In [ ]:
#| eval: false
pipeline = Pipeline([
    ('charngrams', CharNgramVectorizer(ngram_range = (2, 2), lowercase = False, max_features=1000)),
    ('classifier', LogisticRegression(max_iter = 5000, random_state=55))
], verbose=True)

display(pipeline)

Pipeline(steps=[('charngrams',
                 CharNgramVectorizer(lowercase=False, max_features=1000)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=55))],
         verbose=True)

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ........ (step 1 of 2) Processing charngrams, total=   0.9s
[Pipeline] ........ (step 2 of 2) Processing classifier, total=   0.9s


In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

   hemingway      0.927     0.937     0.932       504
       woolf      0.934     0.924     0.929       488

    accuracy                          0.930       992
   macro avg      0.931     0.930     0.930       992
weighted avg      0.930     0.930     0.930       992



In this example the ngram range and max_features are adjusted to extract more ngrams of varying lengths. However, only 500 features are used as features for classification (i.e. half the number used in the examples above) by selecting features based on mutual information scores.  

In [ ]:
#| eval: false
pipeline = Pipeline([
    ('charngrams', CharNgramVectorizer(ngram_range = (2, 4), lowercase = False, max_features=20000)),
	('selector', SelectKBest(score_func=mutual_info_classif, k=500)),
    ('classifier', LogisticRegression(max_iter = 5000, random_state=55))
], verbose=True)

display(pipeline)

Pipeline(steps=[('charngrams',
                 CharNgramVectorizer(lowercase=False, max_features=20000,
                                     ngram_range=(2, 4))),
                ('selector',
                 SelectKBest(k=500,
                             score_func=<function mutual_info_classif>)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=55))],
         verbose=True)

In [ ]:
#| eval: false
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

[Pipeline] ........ (step 1 of 3) Processing charngrams, total=   3.3s
[Pipeline] .......... (step 2 of 3) Processing selector, total=  27.0s
[Pipeline] ........ (step 3 of 3) Processing classifier, total=   1.8s


In [ ]:
#| eval: false
print(classification_report(y_test, y_pred, labels = target_classes, target_names = target_names, digits=3))
plot_confusion_matrix(y_test, y_pred, target_classes, target_names)

              precision    recall  f1-score   support

   hemingway      0.938     0.933     0.935       504
       woolf      0.931     0.936     0.934       488

    accuracy                          0.934       992
   macro avg      0.934     0.935     0.934       992
weighted avg      0.934     0.934     0.934       992



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()